# Classifier Evaluation

Calls `/classify` for each fixture and measures:
- **Out-of-scope rejection recall** — blacklisted articles flagged as `out_of_scope=True`
- **False rejection rate** — in-scope articles incorrectly flagged as `out_of_scope=True`
- **Topic label F1** — precision and recall on expected topic labels

The most important metric is **rejection recall = 1.0** (never miss an out-of-scope article).  
False rejections are less damaging but should stay below 0.10.

**Prerequisite:** NLP service running at `http://localhost:8001` (`/readyz` returns 200).

In [ ]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, NLP_BASE_URL

cases = load_fixture('classify_cases.json')
in_scope = [c for c in cases if not c['expected_out_of_scope']]
out_of_scope = [c for c in cases if c['expected_out_of_scope']]
print(f'Loaded {len(cases)} cases: {len(in_scope)} in-scope, {len(out_of_scope)} blacklisted')

In [ ]:
r = requests.get(f'{NLP_BASE_URL}/readyz')
assert r.status_code == 200, 'Service not ready'

In [ ]:
results = []

for case in cases:
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/classify',
        json={'article_id': case['article_id'], 'text': case['text']}
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, f"{case['id']}: HTTP {resp.status_code}"
    data = resp.json()

    expected_oos = case['expected_out_of_scope']
    returned_oos = data['out_of_scope']
    oos_correct = expected_oos == returned_oos

    expected_topics = set(case.get('expected_topics_include', []))
    returned_topics = set(data['topics'])
    topic_hits = expected_topics & returned_topics
    topic_precision = len(topic_hits) / len(returned_topics) if returned_topics else (1.0 if not expected_topics else 0.0)
    topic_recall = len(topic_hits) / len(expected_topics) if expected_topics else 1.0

    icon = '✅' if oos_correct else '❌'
    label = 'OOS' if expected_oos else 'IN'
    results.append({
        'id': case['id'],
        'description': case['description'],
        'expected_oos': expected_oos,
        'returned_oos': returned_oos,
        'oos_correct': oos_correct,
        'topics': list(returned_topics),
        'topic_precision': topic_precision,
        'topic_recall': topic_recall,
        'latency_s': latency,
    })

    print(f"{icon} [{label}] {case['id']}: out_of_scope={returned_oos} topics={list(returned_topics)}")
    if not oos_correct:
        print(f"   ⚠ WRONG: expected out_of_scope={expected_oos} — {case['description']}")
    print()

In [ ]:
# Rejection metrics
oos_results = [r for r in results if r['expected_oos']]
in_results  = [r for r in results if not r['expected_oos']]

rejection_recall = sum(1 for r in oos_results if r['returned_oos']) / len(oos_results) if oos_results else 1.0
false_rejection_rate = sum(1 for r in in_results if r['returned_oos']) / len(in_results) if in_results else 0.0

# Topic metrics (in-scope only)
in_with_topics = [r for r in in_results if r['topic_recall'] is not None]
avg_topic_prec = sum(r['topic_precision'] for r in in_with_topics) / len(in_with_topics) if in_with_topics else 0.0
avg_topic_rec  = sum(r['topic_recall']    for r in in_with_topics) / len(in_with_topics) if in_with_topics else 0.0
topic_f1 = (2 * avg_topic_prec * avg_topic_rec / (avg_topic_prec + avg_topic_rec)
            if avg_topic_prec + avg_topic_rec > 0 else 0.0)

avg_lat = sum(r['latency_s'] for r in results) / len(results)

print_scorecard('CLASSIFIER', {
    'Cases (total)': len(results),
    'Out-of-scope cases': len(oos_results),
    'In-scope cases': len(in_results),
    'Rejection recall (target=1.0)': rejection_recall,
    'False rejection rate (target<0.10)': false_rejection_rate,
    'Topic label precision (in-scope)': avg_topic_prec,
    'Topic label recall (in-scope)': avg_topic_rec,
    'Topic F1 (target≥0.70)': topic_f1,
    'Average latency (s)': avg_lat,
})

if rejection_recall < 1.0:
    missed = [r['id'] for r in oos_results if not r['returned_oos']]
    print(f'⚠ Missed OOS cases: {missed}')
    print('  → Lower relevance_threshold in config/topics.yaml')

## Tuning Guide

The key parameter is `relevance_threshold` in `config/topics.yaml`.

| Symptom | Direction | Action |
|---------|-----------|--------|
| OOS articles pass through (`rejection_recall < 1.0`) | Too permissive | **Raise** `relevance_threshold` (e.g. 0.40 → 0.50) |
| In-scope articles rejected (`false_rejection_rate > 0.10`) | Too strict | **Lower** `relevance_threshold` (e.g. 0.40 → 0.30) |
| Edge cases hard to separate | Wrong hypothesis | Rewrite `relevance_hypothesis` in `topics.yaml` to be more specific |
| Topic labels wrong / missing | Score threshold too high | Lower `score_threshold` in `topics.yaml` (default 0.5) |
| Too many noisy topic labels | Score threshold too low | Raise `score_threshold` or lower `top_k` |

After changing `topics.yaml`, restart the NLP service (taxonomy is cached on startup) and re-run this notebook.

**Binary search approach for `relevance_threshold`:**
1. Collect all scores by running the cells above with `relevance_threshold: 0.0` (pass everything through)
2. Print scores for OOS cases — the threshold should be just above the highest in-scope score
3. Start at the midpoint and iterate until `rejection_recall=1.0` and `false_rejection_rate<0.10`

In [ ]:
# Helper: show raw relevance scores to aid threshold selection
# Set relevance_threshold: 0.0 in topics.yaml and restart service, then run this cell.

from _scorecard import NLP_BASE_URL
import requests

# This re-calls classify but inspects scores (scores dict includes the relevance hypothesis score
# only if it was added as a label — for inspection, you can add it to labels temporarily).
# Alternative: call the NLI model directly for the relevance hypothesis text.
print('Score distribution analysis:')
print('  Run with relevance_threshold=0.0 in topics.yaml to see all articles pass,')
print('  then inspect topic scores to find the natural separation point.')